# LeetCode 1386 — Cinema Seat Allocation bằng Python

Link đề: https://leetcode.com/problems/cinema-seat-allocation/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra mỗi hàng là **độc lập** và chỉ có **3 khung** (2–5, 4–7, 6–9) ⇒ mỗi hàng chứa tối đa **2** nhóm.
2. Lập luận tham lam đúng cho một hàng: 2 nhóm ⇔ cả hai khung biên trống; nếu không thì 1 nhóm ⇔ có ít nhất một khung trống.
3. Xử lý `n` tới `10^9`: **chỉ duyệt các hàng có ghế bị đặt**, các hàng còn lại đóng góp `2` mỗi hàng ⇒ công thức `2·(n − số hàng bị đặt) + phần còn lại`.
4. Dùng **bitmask** để kiểm tra "khung có ghế nào bị đặt không" bằng một phép `&`.
5. Viết brute force thật sự (thử mọi tổ hợp khung không chồng nhau) để kiểm chứng lời giải tham lam.

## 2. Đề bài & ví dụ

Rạp có `n` hàng ghế, đánh số `1..n`; mỗi hàng có 10 ghế, đánh số `1..10`. Mảng `reservedSeats[i] = [row_i, seat_i]` cho biết ghế `seat_i` ở hàng `row_i` đã có người đặt.

Một nhóm 4 người phải ngồi 4 ghế **cùng hàng**, theo một trong ba khung: ghế **2–5**, ghế **4–7**, hoặc ghế **6–9** (lối đi nằm giữa ghế 3–4 và 7–8; khung 4–7 được phép vì lối đi chia đôi nhóm thành 2 + 2). Một khung chỉ dùng được nếu **không** ghế nào trong khung bị đặt; mỗi ghế chỉ thuộc tối đa một nhóm. Trả về số nhóm 4 người **tối đa** có thể xếp.

- **Ví dụ 1:** `n = 3, reservedSeats = [[1,2],[1,3],[1,8],[2,6],[3,1],[3,10]]` → `4`.
- **Ví dụ 2:** `n = 2, reservedSeats = [[2,1],[1,8],[2,6]]` → `2`.
- **Ví dụ 3:** `n = 4, reservedSeats = [[4,3],[1,4],[4,6],[1,7]]` → `4`.

**Ràng buộc:** `1 <= n <= 10^9`; `1 <= reservedSeats.length <= min(10·n, 10^4)`; `1 <= row_i <= n`; `1 <= seat_i <= 10`; các cặp đều khác nhau.

In [1]:
import random
import time
from collections import defaultdict
from itertools import combinations

EXAMPLES = [
    (3, [[1, 2], [1, 3], [1, 8], [2, 6], [3, 1], [3, 10]], 4),
    (2, [[2, 1], [1, 8], [2, 6]], 2),
    (4, [[4, 3], [1, 4], [4, 6], [1, 7]], 4),
]
BLOCKS = [(2, 5), (4, 7), (6, 9)]
for n, rs, want in EXAMPLES:
    print(f"n={n}, đặt={rs} -> mong đợi {want}")

n=3, đặt=[[1, 2], [1, 3], [1, 8], [2, 6], [3, 1], [3, 10]] -> mong đợi 4
n=2, đặt=[[2, 1], [1, 8], [2, 6]] -> mong đợi 2
n=4, đặt=[[4, 3], [1, 4], [4, 6], [1, 7]] -> mong đợi 4


## 3. Kiến thức nền

### 3.1 Nhóm dữ liệu theo khóa: `defaultdict(set)`

Gom các ghế bị đặt theo hàng: `rows[r].add(s)`. `defaultdict(set)` tự tạo tập rỗng cho khóa mới. Số hàng bị ảnh hưởng là `len(rows)` — tối đa `10^4`, dù `n` có thể tới `10^9`.

In [2]:
rows = defaultdict(set)
for r, s in EXAMPLES[0][1]:
    rows[r].add(s)
print(dict(rows), "| số hàng có ghế bị đặt:", len(rows))

{1: {8, 2, 3}, 2: {6}, 3: {1, 10}} | số hàng có ghế bị đặt: 3


### 3.2 Kiểm tra giao của tập với một đoạn: `any` + `range`

Khung `(a, b)` trống ⇔ không ghế nào trong `a..b` thuộc tập bị đặt: `not any(s in taken for s in range(a, b + 1))`, hoặc dùng `isdisjoint`.

In [3]:
taken = {2, 3, 8}
for a, b in BLOCKS:
    print(f"khung {a}-{b} trống?", taken.isdisjoint(range(a, b + 1)))

khung 2-5 trống? False
khung 4-7 trống? True
khung 6-9 trống? False


### 3.3 Bitmask: biểu diễn tập ghế bằng một số nguyên

Ghế `s` ↔ bit `1 << s`. Tập ghế bị đặt của một hàng là một số `mask`. Khung `(a, b)` có mặt nạ `M = Σ (1 << s), s ∈ [a, b]`. Khung trống ⇔ `mask & M == 0`. Phép `&` trên số nguyên là `O(1)`.

In [4]:
def block_mask(a, b):
    return sum(1 << s for s in range(a, b + 1))

LEFT, MID, RIGHT = block_mask(2, 5), block_mask(4, 7), block_mask(6, 9)
print(f"LEFT  = {LEFT:011b}  (bit 10..0)")
print(f"MID   = {MID:011b}")
print(f"RIGHT = {RIGHT:011b}")
mask = (1 << 2) | (1 << 3) | (1 << 8)
print(f"mask  = {mask:011b} -> LEFT trống? {mask & LEFT == 0}, MID trống? {mask & MID == 0}, RIGHT trống? {mask & RIGHT == 0}")

LEFT  = 00000111100  (bit 10..0)
MID   = 00011110000
RIGHT = 01111000000
mask  = 00100001100 -> LEFT trống? False, MID trống? True, RIGHT trống? False


### 3.4 Sắp xếp danh sách cặp và gom nhóm tuần tự

`sorted(reservedSeats)` sắp theo hàng rồi theo ghế; các ghế cùng hàng đứng liền nhau ⇒ gom nhóm bằng một lượt quét, không cần dict. (Cũng có `itertools.groupby`.)

In [5]:
from itertools import groupby
rs = sorted(EXAMPLES[0][1])
for r, grp in groupby(rs, key=lambda p: p[0]):
    print("hàng", r, "->", [s for _, s in grp])

hàng 1 -> [2, 3, 8]
hàng 2 -> [6]
hàng 3 -> [1, 10]


## 4. Ý tưởng nền tảng / trực giác

Một hàng (lối đi ký hiệu `|`): `1 2 3 | 4 5 6 7 | 8 9 10`.

- Khung **trái** 2–5 và khung **phải** 6–9 **không chồng** nhau ⇒ có thể dùng cả hai ⇒ tối đa 2 nhóm.
- Khung **giữa** 4–7 chồng với cả hai khung kia (chung ghế 4,5 với trái; 6,7 với phải) ⇒ nếu dùng giữa thì chỉ được 1 nhóm.
- Ghế 1 và 10 **không thuộc khung nào** ⇒ đặt ghế 1 hay 10 không ảnh hưởng.

Tính tay Ví dụ 1:

| Hàng | Ghế bị đặt | Trái 2–5 | Giữa 4–7 | Phải 6–9 | Số nhóm |
|---|---|---|---|---|---|
| 1 | 2, 3, 8 | ✗ | ✓ | ✗ | 1 |
| 2 | 6 | ✓ | ✗ | ✗ | 1 |
| 3 | 1, 10 | ✓ | ✓ | ✓ | 2 (trái + phải) |

Tổng 4. Quy luật cho một hàng: **trái và phải cùng trống ⇒ 2; ngược lại có khung nào trống ⇒ 1; không thì 0.**

## 5. Cách 1 — Brute Force / baseline: mảng ghế từng hàng (chỉ với `n` nhỏ)

**Ý tưởng:** tạo lưới `n × 11` (bỏ cột 0), đánh dấu ghế bị đặt; với **mỗi** hàng, **thử mọi tổ hợp** khung đôi một không chồng nhau, lấy tổ hợp lớn nhất. Không dùng tham lam — dùng làm chuẩn kiểm chứng.

In [6]:
def best_row_bruteforce(taken):
    """Thử mọi tập con của 3 khung: các khung phải trống và đôi một không chồng ghế."""
    best = 0
    for k in range(1, 4):
        for combo in combinations(BLOCKS, k):
            seats = [s for a, b in combo for s in range(a, b + 1)]
            if len(seats) == len(set(seats)) and not any(s in taken for s in seats):
                best = max(best, k)
    return best

def families_grid(n, reservedSeats):
    grid = [[False] * 11 for _ in range(n + 1)]
    for r, s in reservedSeats:
        grid[r][s] = True
    total = 0
    for r in range(1, n + 1):
        taken = {s for s in range(1, 11) if grid[r][s]}
        total += best_row_bruteforce(taken)
    return total

for n, rs, want in EXAMPLES:
    print(families_grid(n, rs), "(mong đợi", want, ")")

4 (mong đợi 4 )
2 (mong đợi 2 )
4 (mong đợi 4 )


**Độ phức tạp:** `O(n·10)` thời gian và bộ nhớ (mỗi hàng thử ≤ 7 tổ hợp — hằng số).

**Nút thắt:** `n` tới `10^9` ⇒ lưới `10^{10}` ô — không thể tạo, và duyệt `10^9` hàng trong Python mất hàng chục phút. Nhưng chỉ có tối đa `10^4` hàng có ghế bị đặt; **mọi hàng còn lại trống hoàn toàn và luôn chứa đúng 2 nhóm**. Không cần duyệt chúng.

In [7]:
for n in [5_000, 20_000, 50_000]:
    t0 = time.perf_counter()
    families_grid(n, [[1, 5]])
    print(f"n={n:7d}: {time.perf_counter() - t0:.3f}s  -> ngoại suy n=10^9: ~{(time.perf_counter() - t0) * 1e9 / n / 60:.0f} phút")

n=   5000: 0.028s  -> ngoại suy n=10^9: ~93 phút
n=  20000: 0.112s  -> ngoại suy n=10^9: ~93 phút


n=  50000: 0.291s  -> ngoại suy n=10^9: ~97 phút


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách hàng):** các hàng độc lập ⇒ đáp án = tổng số nhóm tối đa của từng hàng.

**Insight 2 (hàng trống):** hàng không có ghế bị đặt (hoặc chỉ bị đặt ghế 1/10) cho đúng 2 nhóm. Gọi `R` = tập hàng xuất hiện trong `reservedSeats`. Khi đó:

> **đáp án = 2·(n − |R|) + Σ_{r ∈ R} best(r)**

**Insight 3 (tham lam trong một hàng):**
- Nếu trái **và** phải trống ⇒ `best = 2` (không thể hơn vì mọi cặp khung khác đều chồng).
- Ngược lại, nếu trái **hoặc** giữa **hoặc** phải trống ⇒ `best = 1`.
- Ngược lại ⇒ `0`.

**Quy tắc "nếu… thì…" cần nhớ:** *không* được cộng riêng từng khung trống (giữa chồng với hai bên!) — phải xét **đúng thứ tự**: 2 trước, rồi mới tới 1.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ xét hàng có ghế bị đặt (dict các tập)

In [8]:
def best_row(taken):
    free = [taken.isdisjoint(range(a, b + 1)) for a, b in BLOCKS]   # [trái, giữa, phải]
    if free[0] and free[2]:
        return 2
    return 1 if any(free) else 0

def families_sets(n, reservedSeats):
    rows = defaultdict(set)
    for r, s in reservedSeats:
        rows[r].add(s)
    return 2 * (n - len(rows)) + sum(best_row(t) for t in rows.values())

for n, rs, want in EXAMPLES:
    print(families_sets(n, rs), "(mong đợi", want, ")")

4 (mong đợi 4 )
2 (mong đợi 2 )
4 (mong đợi 4 )


Code nộp LeetCode (Cách 2):

In [9]:
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        rows = defaultdict(set)
        for r, s in reservedSeats:
            rows[r].add(s)
        ans = 2 * (n - len(rows))
        for taken in rows.values():
            left = taken.isdisjoint((2, 3, 4, 5))
            mid = taken.isdisjoint((4, 5, 6, 7))
            right = taken.isdisjoint((6, 7, 8, 9))
            if left and right:
                ans += 2
            elif left or mid or right:
                ans += 1
        return ans

print([Solution().maxNumberOfFamilies(n, rs) for n, rs, _ in EXAMPLES])

[4, 2, 4]


**Độ phức tạp Cách 2:** `O(m)` với `m = len(reservedSeats)` (mỗi hàng kiểm tra 3 khung × ≤ 4 ghế = hằng số); bộ nhớ `O(m)`. **Không phụ thuộc `n`.**

### 7.2 Cách 3 — Bitmask 3 khung

Thay tập bằng một số nguyên cho mỗi hàng; mỗi khung kiểm tra bằng một phép `&`. Có thể bỏ qua ghế 1 và 10 ngay khi đọc (chúng không ảnh hưởng) — nhưng **vẫn phải** tính hàng đó là "đã xét" nếu ghi nó vào dict; cách gọn: chỉ ghi ghế 2..9.

In [10]:
LEFT, MID, RIGHT = 0b0000111100, 0b0011110000, 0b1111000000      # bit s ứng với ghế s

def families_bitmask(n, reservedSeats):
    rows = {}
    for r, s in reservedSeats:
        if 2 <= s <= 9:                       # ghế 1 và 10 không ảnh hưởng khung nào
            rows[r] = rows.get(r, 0) | (1 << s)
    ans = 2 * (n - len(rows))
    for mask in rows.values():
        if mask & LEFT == 0 and mask & RIGHT == 0:
            ans += 2
        elif mask & LEFT == 0 or mask & MID == 0 or mask & RIGHT == 0:
            ans += 1
    return ans

assert LEFT == block_mask(2, 5) and MID == block_mask(4, 7) and RIGHT == block_mask(6, 9)
for n, rs, want in EXAMPLES:
    print(families_bitmask(n, rs), "(mong đợi", want, ")")

4 (mong đợi 4 )
2 (mong đợi 2 )
4 (mong đợi 4 )


Lưu ý tinh tế: với cách "chỉ ghi ghế 2..9", một hàng chỉ có ghế 1/10 bị đặt **không** vào dict ⇒ được tính trong phần `2·(n − |R|)` — đúng, vì hàng đó chứa 2 nhóm. Còn hàng đã vào dict mà `mask & LEFT == 0 and mask & RIGHT == 0` thì không thể xảy ra (mọi ghế 2..9 thuộc trái hoặc phải) — nhánh `+2` chỉ để code an toàn.

Code nộp LeetCode (Cách 3):

In [11]:
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        LEFT, MID, RIGHT = 0b0000111100, 0b0011110000, 0b1111000000
        rows = {}
        for r, s in reservedSeats:
            if 2 <= s <= 9:
                rows[r] = rows.get(r, 0) | (1 << s)
        ans = 2 * (n - len(rows))
        for mask in rows.values():
            if not (mask & LEFT) or not (mask & MID) or not (mask & RIGHT):
                ans += 1          # hàng trong dict có ít nhất 1 ghế 2..9 bị đặt => tối đa 1 nhóm
        return ans

print([Solution().maxNumberOfFamilies(n, rs) for n, rs, _ in EXAMPLES])

[4, 2, 4]


**Độ phức tạp Cách 3:** `O(m)` thời gian, `O(số hàng bị đặt)` bộ nhớ; hằng số nhỏ hơn Cách 2 (phép `&` thay vì tạo tập).

### 7.3 Cách 4 — Sắp xếp rồi gom nhóm (không dùng hash)

Hữu ích khi không muốn dùng dict (hoặc ngôn ngữ không có hash tiện lợi): sắp `reservedSeats`, quét tuần tự, mỗi khi đổi hàng thì "chốt" bitmask của hàng trước.

In [12]:
def families_sorted(n, reservedSeats):
    rs = sorted(reservedSeats)
    ans, i, m = 2 * n, 0, len(rs)
    while i < m:
        r, mask = rs[i][0], 0
        while i < m and rs[i][0] == r:
            mask |= 1 << rs[i][1]
            i += 1
        # hàng r: trừ 2 (đã cộng trước) rồi cộng số nhóm thực tế
        free_l, free_m, free_r = mask & LEFT == 0, mask & MID == 0, mask & RIGHT == 0
        ans -= 2
        ans += 2 if (free_l and free_r) else (1 if (free_l or free_m or free_r) else 0)
    return ans

for n, rs, want in EXAMPLES:
    print(families_sorted(n, rs), "(mong đợi", want, ")")

4 (mong đợi 4 )
2 (mong đợi 2 )
4 (mong đợi 4 )


Code nộp LeetCode (Cách 4):

In [13]:
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        LEFT, MID, RIGHT = 0b0000111100, 0b0011110000, 0b1111000000
        rs = sorted(reservedSeats)
        ans, i, m = 2 * n, 0, len(rs)
        while i < m:
            r, mask = rs[i][0], 0
            while i < m and rs[i][0] == r:
                mask |= 1 << rs[i][1]
                i += 1
            fl, fm, fr = not mask & LEFT, not mask & MID, not mask & RIGHT
            ans += (2 if fl and fr else 1 if fl or fm or fr else 0) - 2
        return ans

print([Solution().maxNumberOfFamilies(n, rs) for n, rs, _ in EXAMPLES])

[4, 2, 4]


**Độ phức tạp Cách 4:** `O(m log m)` (sắp xếp) thời gian, `O(m)` bộ nhớ cho bản sao đã sắp.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 và Ví dụ 3. Hàm `trace_bitmask` in mask của từng hàng (bit 9..2), trạng thái 3 khung và số nhóm cộng thêm:

| Ví dụ | Hàng | Ghế 2..9 bị đặt | mask (ghế 9→2) | Trái | Giữa | Phải | + |
|---|---|---|---|---|---|---|---|
| 1 | 1 | 2, 3, 8 | 01000011 | ✗ | ✓ | ✗ | 1 |
| 1 | 2 | 6 | 00010000 | ✓ | ✗ | ✗ | 1 |
| 1 | 3 | (chỉ 1, 10) | – | – | – | – | không vào dict ⇒ thuộc phần 2·(n−\|R\|) |

In [14]:
def trace_bitmask(n, reservedSeats):
    rows = {}
    for r, s in reservedSeats:
        if 2 <= s <= 9:
            rows[r] = rows.get(r, 0) | (1 << s)
    ans = 2 * (n - len(rows))
    print(f"n={n}, số hàng vào dict |R|={len(rows)} -> phần hàng trống: 2*({n}-{len(rows)}) = {ans}")
    for r in sorted(rows):
        mask = rows[r]
        fl, fm, fr = mask & LEFT == 0, mask & MID == 0, mask & RIGHT == 0
        add = 2 if fl and fr else (1 if fl or fm or fr else 0)
        ans += add
        bits = format(mask >> 2, "08b")          # ghế 9..2
        print(f"  hàng {r}: ghế 9..2 = {bits} | trái {'✓' if fl else '✗'} giữa {'✓' if fm else '✗'} phải {'✓' if fr else '✗'} | +{add} -> {ans}")
    return ans

for n, rs, want in [EXAMPLES[0], EXAMPLES[2]]:
    print("kết quả:", trace_bitmask(n, rs), "| mong đợi:", want)
    print()

n=3, số hàng vào dict |R|=2 -> phần hàng trống: 2*(3-2) = 2
  hàng 1: ghế 9..2 = 01000011 | trái ✗ giữa ✓ phải ✗ | +1 -> 3
  hàng 2: ghế 9..2 = 00010000 | trái ✓ giữa ✗ phải ✗ | +1 -> 4
kết quả: 4 | mong đợi: 4

n=4, số hàng vào dict |R|=2 -> phần hàng trống: 2*(4-2) = 4
  hàng 1: ghế 9..2 = 00100100 | trái ✗ giữa ✗ phải ✗ | +0 -> 4
  hàng 4: ghế 9..2 = 00010010 | trái ✗ giữa ✗ phải ✗ | +0 -> 4
kết quả: 4 | mong đợi: 4



## 9. Code nộp LeetCode

In [15]:
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        LEFT, MID, RIGHT = 0b0000111100, 0b0011110000, 0b1111000000   # ghế 2-5, 4-7, 6-9
        rows = {}
        for r, s in reservedSeats:
            if 2 <= s <= 9:                    # ghế 1, 10 không ảnh hưởng
                rows[r] = rows.get(r, 0) | (1 << s)
        ans = 2 * (n - len(rows))              # hàng "sạch": 2 nhóm mỗi hàng
        for mask in rows.values():
            if mask & LEFT == 0 and mask & RIGHT == 0:
                ans += 2
            elif mask & LEFT == 0 or mask & MID == 0 or mask & RIGHT == 0:
                ans += 1
        return ans

for n, rs, want in EXAMPLES:
    got = Solution().maxNumberOfFamilies(n, rs)
    print(got, "OK" if got == want else "SAI")

4 OK
2 OK
4 OK


## 10. Vì sao thuật toán đúng

1. **Độc lập giữa các hàng:** một nhóm chỉ dùng ghế trong một hàng ⇒ tối ưu toàn cục = tổng tối ưu từng hàng.
2. **Mỗi hàng tối đa 2 nhóm, và chỉ theo cặp {trái, phải}:** các cặp khung: (trái, giữa) chung ghế 4–5; (giữa, phải) chung ghế 6–7; (trái, phải) rời nhau. Ba khung cùng lúc thì chắc chắn chồng. Vậy 2 nhóm ⇔ trái và phải đều trống.
3. **Nếu không đạt 2:** đặt được 1 nhóm ⇔ có ít nhất một khung trống. Vậy `best(r)` tính đúng bằng tham lam "2 trước, 1 sau".
4. **Hàng không có ghế 2..9 bị đặt:** cả ba khung trống ⇒ `best = 2`. Các hàng như vậy được tính gộp `2·(n − |R|)` (với `R` là tập hàng có ghế 2..9 bị đặt, hoặc mọi hàng xuất hiện nếu ta không lọc — cả hai đều đúng vì `best` của hàng chỉ có ghế 1/10 cũng bằng 2).

Brute force ở Cách 1 thử **mọi** tổ hợp khung nên là chuẩn; mục 12 kiểm chứng tham lam trùng với nó trên mọi cấu hình của một hàng (`2^10 = 1024` cấu hình).

## 11. Phân tích Time / Space

Gọi `n` = số hàng (≤ `10^9`), `m = len(reservedSeats)` (≤ `10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Lưới + brute force từng hàng | `O(n)` (hằng số ~70) | `O(n)` |
| 2. Dict các tập | `O(m)` | `O(m)` |
| 3. Dict bitmask | `O(m)` | `O(số hàng bị đặt)` |
| 4. Sắp xếp + gom | `O(m log m)` | `O(m)` |

Điểm mấu chốt: lời giải tốt **không phụ thuộc `n`** — `n` chỉ xuất hiện trong phép nhân `2·(n − |R|)`. Python xử lý số nguyên lớn sẵn (ở C++/Java, `2·10^9` vượt `int` 32 bit — cần `long`).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
METHODS = {
    "grid": families_grid,
    "sets": families_sets,
    "bitmask": families_bitmask,
    "sorted": families_sorted,
    "submit": lambda n, rs: Solution().maxNumberOfFamilies(n, rs),
}
edge = [
    (1, [[1, 1], [1, 10]], 2),                 # chỉ ghế 1 và 10: vẫn 2 nhóm
    (1, [[1, 5]], 1),                          # trái và giữa hỏng, phải còn
    (1, [[1, 8]], 1),                          # chỉ trái hoặc giữa (chồng nhau) -> 1
    (1, [[1, 4], [1, 7]], 0),                  # cả ba khung hỏng
    (1, [[1, 2], [1, 9]], 1),                  # chỉ giữa
    (1, [[1, 3], [1, 8]], 1),                  # chỉ giữa
    (5, [[3, 5]], 9),                          # 4 hàng sạch * 2 + 1
    (2, [[1, s] for s in range(1, 11)], 2),    # hàng 1 kín, hàng 2 sạch
]
for name, fn in METHODS.items():
    for n, rs, want in EXAMPLES + edge:
        got = fn(n, rs)
        assert got == want, (name, n, rs, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 11 test


### 12.2 Vét cạn mọi cấu hình của một hàng (`2^10 = 1024`) — tham lam = brute force

In [17]:
for m in range(1 << 10):
    taken = {s + 1 for s in range(10) if m >> s & 1}
    rs = [[1, s] for s in sorted(taken)]
    want = best_row_bruteforce(taken)
    if not rs:
        continue                                     # đề yêu cầu ít nhất 1 ghế bị đặt
    for name, fn in METHODS.items():
        assert fn(1, rs) == want, (name, taken)
print("1023 cấu hình một hàng: mọi cách khớp brute force")

1023 cấu hình một hàng: mọi cách khớp brute force


### 12.3 Ngẫu nhiên nhiều hàng (seed cố định) và test lớn `n = 10^9`

In [18]:
rng = random.Random(1386)
for _ in range(300):
    n = rng.randint(1, 30)
    cells = [(r, s) for r in range(1, n + 1) for s in range(1, 11)]
    k = rng.randint(1, min(10 * n, 40))
    rs = [list(p) for p in rng.sample(cells, k)]
    res = {name: fn(n, rs) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, (n, rs, res)
print("300 rạp ngẫu nhiên: mọi cách khớp nhau")

n_big = 10**9
cells = set()
while len(cells) < 10_000:
    cells.add((rng.randint(1, n_big), rng.randint(1, 10)))
rs_big = [list(p) for p in cells]
for name in ["sets", "bitmask", "sorted", "submit"]:
    t0 = time.perf_counter()
    r = METHODS[name](n_big, rs_big)
    print(f"{name:8s} -> {r} | {time.perf_counter() - t0:.4f}s")

300 rạp ngẫu nhiên: mọi cách khớp nhau
sets     -> 1999991987 | 0.0085s
bitmask  -> 1999991987 | 0.0019s
sorted   -> 1999991987 | 0.0059s
submit   -> 1999991987 | 0.0017s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `if trái: +1; if phải: +1; elif giữa: +1` (đếm chồng khung)

In [19]:
def wrong_elif(n, reservedSeats):
    rows = defaultdict(set)
    for r, s in reservedSeats:
        rows[r].add(s)
    ans = 2 * (n - len(rows))
    for t in rows.values():
        if t.isdisjoint(range(2, 6)):
            ans += 1
        if t.isdisjoint(range(6, 10)):
            ans += 1
        elif t.isdisjoint(range(4, 8)):      # SAI: elif chỉ gắn với điều kiện "phải"
            ans += 1
    return ans

print("hàng chỉ bị đặt ghế 8 -> sai:", wrong_elif(1, [[1, 8]]), "| đúng:", families_sets(1, [[1, 8]]))

hàng chỉ bị đặt ghế 8 -> sai: 2 | đúng: 1


**Vì sao sai:** hàng bị đặt ghế 8: trái trống (+1), phải hỏng ⇒ nhánh `elif` kiểm tra giữa — trống (+1). Tổng 2, nhưng **trái và giữa chồng nhau** ở ghế 4–5 ⇒ thực tế chỉ 1.

**Code đúng:** xét "trái **và** phải" trước (+2), rồi `elif` "trái **hoặc** giữa **hoặc** phải" (+1).

### Lỗi 2 — "Hàng có ghế bị đặt thì tối đa 1 nhóm" (quên ghế 1 và 10 vô hại)

In [20]:
def wrong_any_reserved(n, reservedSeats):
    rows = defaultdict(set)
    for r, s in reservedSeats:
        rows[r].add(s)
    ans = 2 * (n - len(rows))
    for t in rows.values():
        if any(t.isdisjoint(range(a, b + 1)) for a, b in BLOCKS):
            ans += 1                          # SAI: bỏ qua khả năng 2 nhóm
    return ans

for n, rs, want in [EXAMPLES[0], (1, [[1, 1], [1, 10]], 2)]:
    print("sai:", wrong_any_reserved(n, rs), "| đúng:", want)

sai: 3 | đúng: 4
sai: 1 | đúng: 2


**Vì sao sai:** hàng chỉ bị đặt ghế 1 và/hoặc 10 vẫn còn trọn cả trái lẫn phải ⇒ 2 nhóm. Ví dụ 1 có hàng 3 như vậy ⇒ code sai trả 3 thay vì 4.

**Code đúng:** xét trường hợp 2 nhóm; hoặc lọc bỏ ghế 1/10 khi gom (Cách 3) **và** khi đó mới được khẳng định "hàng trong dict tối đa 1 nhóm".

### Lỗi 3 — Lệch bit: mask ghế dùng `1 << (s-1)` nhưng mặt nạ khung dùng `1 << s`

In [21]:
def wrong_shift(n, reservedSeats):
    rows = {}
    for r, s in reservedSeats:
        rows[r] = rows.get(r, 0) | (1 << (s - 1))     # SAI: lệch 1 so với LEFT/MID/RIGHT
    ans = 2 * (n - len(rows))
    for mask in rows.values():
        if mask & LEFT == 0 and mask & RIGHT == 0:
            ans += 2
        elif mask & LEFT == 0 or mask & MID == 0 or mask & RIGHT == 0:
            ans += 1
    return ans

for n, rs, want in EXAMPLES:
    print("sai:", wrong_shift(n, rs), "| đúng:", want)

sai: 2 | đúng: 4
sai: 2 | đúng: 2
sai: 5 | đúng: 4


**Vì sao sai:** ghế `s` bị ghi vào bit `s−1` trong khi `LEFT = 0b0000111100` giả định ghế `s` ở bit `s`. Ghế 6 (bit 5) bị hiểu như ghế 5; ghế 10 (bit 9) bị hiểu như ghế 9... ⇒ kết quả sai ở Ví dụ 1 và 3.

**Code đúng:** thống nhất một quy ước: ghế `s` ↔ bit `s` cho **cả** mask hàng lẫn mặt nạ khung (hoặc cùng dùng `s−1`).

### Lỗi 4 — Duyệt toàn bộ `n` hàng

Kết quả đúng nhưng không thể chạy với `n = 10^9` — ta ước lượng thời gian thay vì chạy thật:

In [22]:
def slow_all_rows(n, reservedSeats):
    rows = defaultdict(set)
    for r, s in reservedSeats:
        rows[r].add(s)
    return sum(best_row(rows.get(r, set())) for r in range(1, n + 1))   # O(n)

t0 = time.perf_counter(); r1 = slow_all_rows(200_000, [[1, 5]]); dt = time.perf_counter() - t0
print(f"n=200000: {r1} trong {dt:.3f}s -> n=10^9 cần khoảng {dt * 5000 / 60:.0f} phút")
print("Cách 3 với n=10^9:", families_bitmask(10**9, [[1, 5]]))

n=200000: 399999 trong 0.091s -> n=10^9 cần khoảng 8 phút
Cách 3 với n=10^9: 1999999999


**Code đúng:** chỉ xét hàng có ghế bị đặt; cộng `2·(n − |R|)` cho phần còn lại.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n` khổng lồ nhưng dữ liệu thưa (`≤ 10^4` mục)** ⇒ chỉ xử lý phần "có dữ liệu", phần còn lại tính bằng **công thức**.
- **Ít trạng thái cố định (10 ghế, 3 khung)** ⇒ bitmask + mặt nạ hằng.
- Trước khi tham lam, **liệt kê quan hệ chồng nhau** giữa các lựa chọn (ở đây: giữa chồng cả hai bên) ⇒ quyết định thứ tự kiểm tra.
- Với cấu hình nhỏ (`2^10`), **vét cạn** để kiểm chứng tham lam — rất rẻ và chắc chắn.
- Nhớ các phần tử "vô hại" (ghế 1, 10) — dễ gây lỗi đếm hàng.

## 15. Biến thể của bài

1. **Nhóm 4 người ngồi 4 ghế liên tiếp bất kỳ** (không ràng buộc lối đi): mỗi đoạn trống dài `L` chứa `L // 4` nhóm (bài tập 5).
2. **Nhóm kích thước `g`, hàng rộng `w`, lối đi tùy ý:** liệt kê khung hợp lệ, mỗi hàng giải bài "chọn nhiều đoạn không giao nhau nhất" (tham lam theo điểm kết thúc).
3. **Truy vấn động:** thêm/hủy đặt ghế và hỏi đáp án sau mỗi thao tác ⇒ giữ `mask` từng hàng và tổng hiện tại, cập nhật `O(1)` mỗi thao tác.
4. **LeetCode 855 — Exam Room**, **1845 — Seat Reservation Manager:** các bài thiết kế xếp chỗ khác.
5. **Đếm số cách xếp** (không chỉ số nhóm tối đa): tổ hợp theo từng hàng.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưới + brute force | mảng ghế, thử mọi tổ hợp khung mỗi hàng | `O(n)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Dict tập | chỉ hàng bị đặt, tham lam 3 khung | `O(m)` | `O(m)` | dễ đọc |
| 3. Dict bitmask | như 2, kiểm tra bằng `&` | `O(m)` | `O(m)` | **lời giải chuẩn**, nhanh |
| 4. Sắp xếp + gom | không hash, quét tuần tự | `O(m log m)` | `O(m)` | không muốn dùng dict |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxNumberOfFamilies(2, [[1,5],[2,2]])` và `maxNumberOfFamilies(10**9, [[7,1]])`?

**Bài 2 (tính tay):** tính `mask` (dạng nhị phân, bit `s` = ghế `s`) của hàng có ghế 3, 6, 10 bị đặt, và cho biết khung nào trống.

**Bài 3 (điền TODO):** hoàn thiện hàm số nhóm tối đa của **một** hàng từ `mask`.

In [23]:
def best_from_mask(mask):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    # TODO: dùng LEFT, MID, RIGHT; 2 nếu trái & phải trống; 1 nếu có khung trống; else 0
    pass

**Bài 4 (tự viết lại):** viết lại lời giải dùng `collections.Counter`/`dict` đếm **số hàng** theo từng `mask` (nhiều hàng có cùng mask chỉ cần tính `best` một lần rồi nhân số lượng).

**Bài 5 (mini challenge):** biến thể **không có lối đi**: một nhóm 4 người ngồi 4 ghế **liên tiếp bất kỳ** trong hàng 10 ghế. Tính số nhóm tối đa với `n` tới `10^9`.

In [24]:
def families_any_four(n, reservedSeats):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Hàng 1 ghế 5: trái, giữa hỏng | hàng 2 ghế 2: giữa và phải còn nhưng chồng nhau | 2 và 2·10^9 |
| 2 | bit 3, 6, 10 | `0b10001001000` | trái ✗ (3), giữa ✗ (6), phải ✗ (6) ⇒ 0 nhóm |
| 3 | `mask & LEFT == 0` | kiểm tra trái & phải trước | `return 2 if ... else 1 if ... else 0` |
| 4 | gom mask theo hàng như Cách 3 | `Counter(rows.values())` | `sum(cnt * best_from_mask(m))` |
| 5 | hàng sạch: `10 // 4 = 2` | tách hàng thành các đoạn trống liên tiếp | cộng `len // 4` mỗi đoạn |

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
ans1 = [families_bitmask(2, [[1, 5], [2, 2]]), families_bitmask(10**9, [[7, 1]])]
print("Bài 1:", ans1)
assert ans1 == [2, 2 * 10**9]

# Bài 2
mask2 = (1 << 3) | (1 << 6) | (1 << 10)
free2 = (mask2 & LEFT == 0, mask2 & MID == 0, mask2 & RIGHT == 0)
print("Bài 2:", bin(mask2), free2)
assert bin(mask2) == "0b10001001000" and free2 == (False, False, False)

Bài 1: [2, 2000000000]
Bài 2: 0b10001001000 (False, False, False)


In [26]:
# Bài 3
def best_from_mask_sol(mask):
    fl, fm, fr = mask & LEFT == 0, mask & MID == 0, mask & RIGHT == 0
    return 2 if fl and fr else 1 if fl or fm or fr else 0

for m in range(1 << 11):
    taken = {s for s in range(1, 11) if m >> s & 1}
    assert best_from_mask_sol(m & ~1) == best_row_bruteforce(taken)
print("Bài 3: OK (2048 mask)")

Bài 3: OK (2048 mask)


In [27]:
# Bài 4
from collections import Counter

def families_counter(n, reservedSeats):
    rows = {}
    for r, s in reservedSeats:
        rows[r] = rows.get(r, 0) | (1 << s)
    cnt = Counter(rows.values())
    return 2 * (n - len(rows)) + sum(c * best_from_mask_sol(m) for m, c in cnt.items())

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 25)
    cells = [(r, s) for r in range(1, n + 1) for s in range(1, 11)]
    rs = [list(p) for p in rng.sample(cells, rng.randint(1, min(10 * n, 30)))]
    assert families_counter(n, rs) == families_grid(n, rs)
print("Bài 4: OK")

Bài 4: OK


In [28]:
# Bài 5
def families_any_four_sol(n, reservedSeats):
    rows = defaultdict(set)
    for r, s in reservedSeats:
        rows[r].add(s)
    ans = 2 * (n - len(rows))                 # 10 // 4 = 2 cho hàng sạch
    for taken in rows.values():
        run = 0
        for s in range(1, 12):                # ghế 11 là "canh" để chốt đoạn cuối
            if s <= 10 and s not in taken:
                run += 1
            else:
                ans += run // 4
                run = 0
    return ans

def any_four_brute_row(taken):
    # quy hoạch động: f[i] = số nhóm tối đa trên ghế 1..i
    f = [0] * 11
    for i in range(1, 11):
        f[i] = f[i - 1]
        if i >= 4 and all(s not in taken for s in range(i - 3, i + 1)):
            f[i] = max(f[i], f[i - 4] + 1)
    return f[10]

for m in range(1 << 10):
    taken = {s + 1 for s in range(10) if m >> s & 1}
    if taken:
        assert families_any_four_sol(1, [[1, s] for s in taken]) == any_four_brute_row(taken)
assert families_any_four_sol(10**9, [[1, 5]]) == 2 * 10**9          # ghế 1-4 và 6-9 vẫn đủ 2 nhóm
assert families_any_four_sol(1, [[1, 3], [1, 8]]) == 1                # đoạn 4-7
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "n lớn, dữ liệu thưa, mỗi đơn vị có ít trạng thái":**

```python
LEFT, MID, RIGHT = 0b0000111100, 0b0011110000, 0b1111000000   # bit s = ghế s
rows = {}
for r, s in reservedSeats:
    if 2 <= s <= 9:
        rows[r] = rows.get(r, 0) | (1 << s)
ans = 2 * (n - len(rows))                    # phần "sạch" tính bằng công thức
for mask in rows.values():
    if not mask & LEFT and not mask & RIGHT: ans += 2
    elif not mask & LEFT or not mask & MID or not mask & RIGHT: ans += 1
return ans
```

**Checklist trước khi submit:**
- [ ] Không duyệt `n` hàng — chỉ hàng có ghế bị đặt.
- [ ] Xét 2 nhóm (trái **và** phải) **trước**, rồi mới 1 nhóm.
- [ ] Không cộng riêng từng khung trống (giữa chồng hai bên).
- [ ] Ghế 1, 10 vô hại: hàng chỉ có ghế 1/10 vẫn được 2 nhóm.
- [ ] Bit của ghế và mặt nạ khung cùng quy ước.
- [ ] Đúng chữ ký: `maxNumberOfFamilies(self, n, reservedSeats) -> int`.